In [ ]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
from pyspark.streaming import StreamingContext
import pyspark.sql.functions as F
import pyspark.sql.types as T
import re
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
STREAM_HOST = os.environ.get("SOCKET_HOST", "localhost")
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("StreamingConcepts").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)
spark

# Spark Streaming Concepts

## What Is a Data Stream?

A **data stream** is an unbounded, continuously arriving sequence of records. Unlike a bounded dataset (a file or database table), a stream has no defined end — records arrive as events happen in the real world.

Streaming data pipelines generally follow a **producer → (bus) → consumer** pattern:

```
┌──────────────────┐       socket / Kafka       ┌──────────────────────┐
│  Producer        │ ──────────────────────────▶ │  Spark Consumer      │
│  (TwitterStreamer│                             │  (SparkStreaming)     │
│   polls X API)   │                             │  extracts mentions,  │
└──────────────────┘                             │  hashtags, saves     │
                                                 └──────────────────────┘
```

## The X/Twitter API as a Canonical Source

Social media feeds are a classic streaming data source for teaching purposes. The X/Twitter API v2 provides a `GET /2/tweets/search/recent` endpoint that returns the most recent tweets matching a query. By polling this endpoint repeatedly and forwarding results to a socket, we can feed a continuous stream of tweets into a Spark consumer.

> **Note**: As of 2023, the X API v2 requires a paid developer account (Basic tier or above). The **Tweet Simulator** in Example 3 lets you run the pipeline offline without credentials.

# Example 1: The X API v2 — REST Call

The `GET /2/tweets/search/recent` endpoint returns up to 100 recent tweets matching a query string. Authentication uses a **Bearer Token** sent in the `Authorization` header.

The Java version (`TwitterStreamer.java`) uses `HttpURLConnection`. The Python equivalent uses the `requests` library.

In [ ]:
# Example API call — requires a valid bearer token to actually run.
# Shown here for reference; replace YOUR_BEARER_TOKEN with a real token.

import requests, json

def search_tweets(query: str, bearer_token: str, max_results: int = 10) -> dict:
    """Call GET /2/tweets/search/recent and return the JSON response."""
    url = (
        "https://api.twitter.com/2/tweets/search/recent"
        f"?query={query}&max_results={max_results}"
    )
    headers = {"Authorization": f"Bearer {bearer_token}"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()   # raises HTTPError on 4xx/5xx
    return response.json()

# Example shape of the response:
sample_response = {
    "data": [
        {"id": "1234567890", "text": "@hanisaf great lecture on #spark today! #uga"},
        {"id": "1234567891", "text": "Deployed @ApacheSpark 4.0 in prod #datapipeline"},
    ],
    "meta": {"newest_id": "1234567891", "oldest_id": "1234567890",
              "result_count": 2, "next_token": "..."}
}
print(json.dumps(sample_response, indent=2))

# Example 2: TwitterStreamer Class — Python Port

`TwitterStreamer.java` does three things:
1. Opens a TCP socket server and waits for Spark to connect.
2. Polls `GET /2/tweets/search/recent` every second.
3. Forwards each tweet JSON object as a newline-delimited string to the socket.

The Python port below is a faithful translation. Save it as `twitter_streamer.py` and run it from a terminal before starting the Spark consumer:

```bash
python3 twitter_streamer.py YOUR_BEARER_TOKEN "#bigdata" 7777
```

In [ ]:
# ── Python port of TwitterStreamer.java ──────────────────────────────────
# Requires an X/Twitter API v2 Bearer Token (paid developer account).
# Run this file from a terminal:
#   python3 twitter_streamer.py <bearer_token> <query> <port>

import socket, time, requests, json, sys
from datetime import datetime

class TwitterStreamer:
    """Poll the X API v2 recent-search endpoint and forward results to a socket."""

    def __init__(self, bearer_token: str, port: int = 7777):
        self.bearer_token = bearer_token
        self.port = port
        self._init_socket()

    def _init_socket(self):
        print("Awaiting client to connect ...")
        server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        server.bind(("localhost", self.port))
        server.listen(1)
        self.conn, addr = server.accept()
        print(f"Spark connected from {addr}.")

    def search(self, query: str, max_results: int = 10) -> dict:
        """Call GET /2/tweets/search/recent and forward each tweet as a JSON line."""
        url = (
            "https://api.twitter.com/2/tweets/search/recent"
            f"?query={query}&max_results={max_results}"
        )
        headers = {"Authorization": f"Bearer {self.bearer_token}"}
        response = requests.get(url, headers=headers)
        data = response.json()
        for tweet in data.get("data", []):
            line = json.dumps(tweet)
            self.conn.sendall((line + "\n").encode())
        return data

    def stream(self, query: str, poll_interval: float = 1.0):
        """Poll indefinitely, forwarding results to the Spark consumer."""
        while True:
            try:
                self.search(query)
            except Exception as e:
                print(f"Error: {e}")
            time.sleep(poll_interval)


if __name__ == "__main__":
    if len(sys.argv) != 4:
        print("Usage: python3 twitter_streamer.py <bearer_token> <query> <port>")
        sys.exit(1)
    token, query, port = sys.argv[1], sys.argv[2], int(sys.argv[3])
    ts = TwitterStreamer(token, port)
    ts.stream(query)

In [ ]:
# Write the TwitterStreamer to a standalone script file.

TWITTER_STREAMER_SRC = (
    "#!/usr/bin/env python3\n"
    "import socket, time, requests, json, sys\n\n"
    "class TwitterStreamer:\n"
    "    def __init__(self, bearer_token, port=7777):\n"
    "        self.bearer_token = bearer_token; self._init_socket(port)\n"
    "    def _init_socket(self, port):\n"
    "        print('Awaiting client ...')\n"
    "        s = socket.socket(); s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)\n"
    "        s.bind(('localhost', port)); s.listen(1)\n"
    "        self.conn, addr = s.accept(); print(f'Connected from {addr}')\n"
    "    def search(self, query, max_results=10):\n"
    "        url = f'https://api.twitter.com/2/tweets/search/recent?query={query}&max_results={max_results}'\n"
    "        r = requests.get(url, headers={'Authorization': f'Bearer {self.bearer_token}'})\n"
    "        for t in r.json().get('data', []):\n"
    "            self.conn.sendall((json.dumps(t)+'\\n').encode())\n"
    "    def stream(self, query, poll_interval=1.0):\n"
    "        while True:\n"
    "            try: self.search(query)\n"
    "            except Exception as e: print(e)\n"
    "            time.sleep(poll_interval)\n"
    "if __name__ == '__main__':\n"
    "    if len(sys.argv) != 4:\n"
    "        print('Usage: twitter_streamer.py <token> <query> <port>'); sys.exit(1)\n"
    "    ts = TwitterStreamer(sys.argv[1], int(sys.argv[3])); ts.stream(sys.argv[2])\n"
)

with open("twitter_streamer.py", "w") as f:
    f.write(TWITTER_STREAMER_SRC)
print("twitter_streamer.py written.")
print("Run from a terminal: python3 twitter_streamer.py <token> <query> 7777")

# Example 3: Tweet Simulator — Offline Testing

When you don't have an X API token, the **Tweet Simulator** sends a rotating list of synthetic tweet-like strings to the same socket port.

**Run this cell first**, then run the Spark consumer cell below.  
The simulator runs in a background daemon thread and stops automatically when you restart the kernel.

In [ ]:
# ── Tweet Simulator ───────────────────────────────────────────────────────
# Sends synthetic tweet-like lines to a socket so the Spark consumer can
# run without a live X API token.  Run this cell BEFORE the consumer cell.

import socket, threading, time, random

TWEETS = [
    "Excited to attend #SIGMOD2025 next week! @databricks talk on streaming #bigdata",
    "Just deployed @ApacheSpark 4.0 in prod #datapipeline #streaming https://spark.apache.org/releases/",
    "@hanisaf great lecture on #java streams today map/filter/reduce clicked #uga #cs",
    "New paper on watermark strategies in event-time streaming https://arxiv.org/abs/2501.00001 #dataengineering @databricks",
    "Watching @GoDawgs game while finishing #Spark homework #uga #football",
    "@ApacheSpark vs @hadoopconference — which session should I attend? #bigdata #conference",
    "Anyone else find @delta_lake game-changing for streaming+batch consistency? #lakehouses #dataengineering",
    "Live keynote stream at https://conf.example.org/live @ApacheKafka topic partitioning deep dive #kafka",
    "Office hours today 3pm #uga #cs students welcome to discuss #streaming homework @hanisaf",
    "The future is unified batch/stream APIs in @ApacheSpark #spark #dataengineering #streaming",
    "Loving the new @ApacheKafka 4.0 features — queue mode is a game changer #kafka #streaming",
    "Reading about Flink vs Spark streaming trade-offs https://blog.example.com/flink-vs-spark #flink #spark",
    "@hanisaf when does the midterm grade post? #uga #anxiety",
    "#Python is taking over #dataengineering — even Java shops are switching @pydata",
    "Structured vs DStream API: which should you choose in 2025? #spark #streaming @ApacheSpark",
]

def run_simulator(port=7777, delay=1.5):
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind(("localhost", port))
    srv.listen(1)
    print(f"Simulator: waiting for Spark to connect on port {port} ...")
    conn, addr = srv.accept()
    print(f"Simulator: Spark connected from {addr}.  Streaming synthetic tweets ...")
    i = 0
    try:
        while True:
            tweet = TWEETS[i % len(TWEETS)]
            conn.sendall((tweet + "\n").encode())
            i += 1
            time.sleep(delay)
    except (BrokenPipeError, ConnectionResetError):
        print("Simulator: Spark disconnected.")
    finally:
        conn.close(); srv.close()

sim_thread = threading.Thread(target=run_simulator, daemon=True)
sim_thread.start()
print("Simulator thread started — now run the Spark consumer cell below.")

# Example 4: Spark Consumer — Port of SparkStreaming.java

`SparkStreaming.java` connects to the socket, then for each 10-second batch:
- Extracts all `@mentions` using `Pattern.compile("@[a-zA-Z0-9_]+").matcher(...)`
- Extracts all `#hashtags` using the same regex approach
- Counts each token by value with `countByValue()`
- Saves mention counts to `out/mentions/` and hashtag counts to `out/hashtags/`
- Prints both counts to the console

The Python port uses `re.findall()` which is equivalent to the Java `findAll()` helper method in the original.

**Run this cell after the simulator cell.**  
Interrupt the kernel (■) to stop the stream.

In [ ]:
# ── Spark Consumer — Python port of SparkStreaming.java ──────────────────
# Reads lines from the socket, extracts @mentions and #hashtags,
# counts each by value, saves results, and prints per batch.
#
# Run this cell AFTER starting the simulator (or TwitterStreamer).
# Interrupt the kernel (■) to stop the stream.

import re

ssc = StreamingContext(sc, 10)   # 10-second batch interval

lines = ssc.socketTextStream("localhost", 7777)

# Extract @mentions and #hashtags using regex  (mirrors SparkStreaming.java)
mentions = lines.flatMap(lambda line: re.findall(r"@[a-zA-Z0-9_]+", line))
hashtags = lines.flatMap(lambda line: re.findall(r"#[a-zA-Z0-9_]+", line))

mention_counts = mentions.countByValue()
hashtag_counts = hashtags.countByValue()

mention_counts.dstream().saveAsTextFiles("out/mentions", "txt")
mention_counts.pprint()

hashtag_counts.dstream().saveAsTextFiles("out/hashtags", "txt")
hashtag_counts.pprint()

print("Streaming started — interrupt the kernel to stop.")
ssc.start()
ssc.awaitTermination()
ssc.stop(stopSparkContext=False)

In [ ]:
spark.stop()